# DGT Baseline — Differential Graph Transformer (Stanford CS224W)

**Purpose:** Proposal-promised baseline. DGT uses Pearson correlation for dynamic edge generation (purely statistical graph). We compare against our WIRE model which uses multi-relational fundamental graphs.

**Architecture (adapted from Kevin Xiang Li, CS224W):**
- Temporal causal attention (per stock, across lookback window)
- Spatial differential attention (across stocks, conditioned on Pearson correlation matrix)
- Differential attention: softmax(Q₁K₁ᵀ)·A - λ·softmax(Q₂K₂ᵀ)

**Evaluation:** Same as v3.0 — ic_loss, rank-normalized targets, Rank IC, paired t-test

**Config:** N=300, 25 features, LOOKBACK=20, HORIZON=5, 20-seed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = "/content/drive/MyDrive/2026 Spring/STAT 3106/3106_Projects/Projects/output/v_dgt_baseline"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output → {OUT_DIR}")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr, ttest_rel, rankdata
import math, json, time, warnings
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}, PyTorch: {torch.__version__}")

## 1. Data Loading & Feature Engineering

Identical to v3.0 — same 300 stocks, 25 features, same train/val/test split.

In [ ]:
DATA = "/content/drive/MyDrive/2026 Spring/STAT 3106/3106_Projects/Projects/Data"
crsp = pd.read_csv(f"{DATA}/qf0egyr4ffi0pszj.csv", parse_dates=["DlyCalDt"])
crsp = crsp.sort_values(["PERMNO", "DlyCalDt"]).reset_index(drop=True)
compustat = pd.read_csv(f"{DATA}/gg3axrtvut5hi5hh.csv", parse_dates=["datadate"])
compustat = compustat.rename(columns={"LPERMNO": "PERMNO"})
print(f"CRSP: {crsp.shape[0]:,} rows, {crsp['PERMNO'].nunique()} stocks")
print(f"Compustat: {compustat.shape[0]:,} rows, {compustat['PERMNO'].nunique()} stocks")

In [ ]:
N_TARGET = 300
date_range = crsp["DlyCalDt"].nunique()
stock_counts = crsp.groupby("PERMNO")["DlyCalDt"].count()
valid_permnos = stock_counts[stock_counts >= date_range * 0.8].index
gsector_map = compustat.drop_duplicates("PERMNO", keep="last").set_index("PERMNO")["gsector"]
valid_permnos = valid_permnos[valid_permnos.isin(gsector_map.index)]
avg_vol = crsp[crsp["PERMNO"].isin(valid_permnos)].groupby("PERMNO")["DlyVol"].mean()
top_permnos = sorted(avg_vol.nlargest(N_TARGET).index.tolist())
crsp_sub = crsp[crsp["PERMNO"].isin(top_permnos)].copy()
permno_to_idx = {p: i for i, p in enumerate(top_permnos)}
N_STOCKS = len(top_permnos)
sectors = np.array([gsector_map.get(p, -1) for p in top_permnos])
print(f"Selected {N_STOCKS} stocks, {len(crsp_sub):,} daily records")

In [ ]:
print("Computing technical indicators per stock...")
t0 = time.time()

def add_technicals(df):
    results = []
    total = df["PERMNO"].nunique()
    for i, (permno, g) in enumerate(df.groupby("PERMNO")):
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{total} stocks...")
        g = g.sort_values("DlyCalDt").copy()
        c, h, l, v, r = g["DlyPrc"], g["DlyHigh"], g["DlyLow"], g["DlyVol"], g["DlyRet"]
        cs = c.replace(0, np.nan)
        delta = c.diff()
        avg_gain = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
        avg_loss = (-delta).clip(lower=0).ewm(span=14, adjust=False).mean()
        g["RSI"] = (100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan))) / 100
        ema12 = c.ewm(span=12, adjust=False).mean()
        ema26 = c.ewm(span=26, adjust=False).mean()
        macd = ema12 - ema26
        g["MACD_hist"] = (macd - macd.ewm(span=9, adjust=False).mean()) / cs
        sma20 = c.rolling(20, min_periods=10).mean()
        std20 = c.rolling(20, min_periods=10).std()
        g["BB_pctB"] = (c - (sma20 - 2 * std20)) / (4 * std20).replace(0, np.nan)
        tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
        g["ATR_norm"] = tr.rolling(14, min_periods=7).mean() / cs
        for w in [5, 20, 60]:
            sma = c.rolling(w, min_periods=max(1, w // 2)).mean()
            g[f"close_sma{w}"] = c / sma.replace(0, np.nan)
        for w in [5, 20]:
            g[f"mom_{w}d"] = c.pct_change(w)
        for w in [5, 20]:
            g[f"rvol_{w}d"] = r.rolling(w, min_periods=max(1, w // 2)).std() * np.sqrt(252)
        g["vol_ratio"] = v / v.rolling(20, min_periods=10).mean().replace(0, np.nan)
        g["log_vol_chg"] = np.log1p(v).diff()
        g["ba_spread"] = (g["DlyAsk"] - g["DlyBid"]) / cs
        g["turnover"] = v / (g["ShrOut"].replace(0, np.nan) * 1000)
        g["excess_ret"] = r - g["sprtrn"]
        results.append(g)
    return pd.concat(results, ignore_index=True)

crsp_sub = add_technicals(crsp_sub)
print(f"Done in {time.time() - t0:.1f}s — {len(crsp_sub):,} rows")

In [ ]:
TECH_COLS = ["RSI", "MACD_hist", "BB_pctB", "ATR_norm",
             "close_sma5", "close_sma20", "close_sma60",
             "mom_5d", "mom_20d", "rvol_5d", "rvol_20d",
             "vol_ratio", "log_vol_chg", "ba_spread", "turnover", "excess_ret"]
FUND_COLS = ["roe", "leverage", "profit_margin", "log_assets"]
FEATURE_NAMES = (["close", "open_close", "high_close", "low_close", "log_vol", "return"]
                 + TECH_COLS + FUND_COLS)

def build_feature_tensor(df, permnos, p2i):
    dates = sorted(df["DlyCalDt"].unique())
    T, N = len(dates), len(permnos)
    F = 6 + len(TECH_COLS)
    feat = np.full((T, N, F), np.nan, dtype=np.float32)
    d2t = {d: t for t, d in enumerate(dates)}
    df = df.copy()
    df["_t"] = df["DlyCalDt"].map(d2t)
    df["_n"] = df["PERMNO"].map(p2i)
    df = df.dropna(subset=["_t", "_n"])
    t = df["_t"].astype(int).values
    n = df["_n"].astype(int).values
    c = df["DlyPrc"].values.astype(np.float32)
    cs = np.where(c == 0, 1.0, c)
    feat[t, n, 0] = c
    feat[t, n, 1] = df["DlyOpen"].values / cs
    feat[t, n, 2] = df["DlyHigh"].values / cs
    feat[t, n, 3] = df["DlyLow"].values / cs
    feat[t, n, 4] = np.log1p(df["DlyVol"].values)
    feat[t, n, 5] = df["DlyRet"].values
    for i, col in enumerate(TECH_COLS):
        feat[t, n, 6 + i] = df[col].values
    return feat, dates

features, dates = build_feature_tensor(crsp_sub, top_permnos, permno_to_idx)
print(f"Base tensor: {features.shape}")

fund = compustat[["PERMNO", "datadate", "rdq", "atq", "ceqq", "niq"]].copy()
fund["rdq"] = pd.to_datetime(fund["rdq"])
fund["avail_date"] = fund["rdq"].fillna(fund["datadate"] + pd.Timedelta(days=45))
def safe_ratio(num, denom, clip=10):
    return (num / denom.replace(0, np.nan)).clip(-clip, clip)
fund["roe"] = safe_ratio(fund["niq"], fund["ceqq"])
fund["leverage"] = safe_ratio(fund["atq"], fund["ceqq"])
fund["profit_margin"] = safe_ratio(fund["niq"], fund["atq"])
fund["log_assets"] = np.log1p(fund["atq"].clip(lower=0))

T, N, F_base = features.shape
fund_feat = np.full((T, N, len(FUND_COLS)), np.nan, dtype=np.float32)
dates_arr = np.array(dates)
print("Merging fundamentals...")
for permno in top_permnos:
    ni = permno_to_idx[permno]
    sf = fund[fund["PERMNO"] == permno].sort_values("avail_date")
    for _, row in sf.iterrows():
        mask = dates_arr >= row["avail_date"]
        if mask.any():
            fund_feat[mask, ni, :] = [row[c] for c in FUND_COLS]
features = np.concatenate([features, fund_feat], axis=2)

print("Filling NaN...")
for n in range(N):
    df_tmp = pd.DataFrame(features[:, n, :])
    features[:, n, :] = df_tmp.ffill().bfill().values

clip_map = {
    6: (0,1), 7: (-0.1,0.1), 8: (-1,2), 9: (0,0.5),
    10: (0.5,2.0), 11: (0.5,2.0), 12: (0.5,2.0),
    13: (-0.5,0.5), 14: (-0.5,0.5), 15: (0,2.0), 16: (0,2.0),
    17: (0,10.0), 19: (-0.01,0.1), 20: (0,0.1), 21: (-0.2,0.2),
}
for idx, (lo, hi) in clip_map.items():
    features[:, :, idx] = np.clip(features[:, :, idx], lo, hi)
features = np.nan_to_num(features, nan=0.0)
INPUT_DIM = features.shape[2] - 1
print(f"Final tensor: {features.shape}, INPUT_DIM={INPUT_DIM}")

## 2. Correlation Graph & Dataset

DGT uses Pearson correlation for spatial attention conditioning. Same MP denoising + precompute cache as v3.0.

In [ ]:
LOOKBACK = 20
HORIZON = 5
CORR_WINDOW = 60
CORR_REUSE = 5

def mp_denoise(corr, T, N):
    eigenvalues, eigenvectors = np.linalg.eigh(corr)
    gamma = N / T
    lambda_plus = (1 + np.sqrt(gamma)) ** 2
    signal_mask = eigenvalues > lambda_plus
    n_signal = signal_mask.sum()
    if 0 < n_signal < N:
        signal_sum = eigenvalues[signal_mask].sum()
        noise_val = max((N - signal_sum) / (N - n_signal), 0.0)
        eigenvalues[~signal_mask] = noise_val
    corr_denoised = (eigenvectors * eigenvalues) @ eigenvectors.T
    d = np.sqrt(np.maximum(np.diag(corr_denoised), 1e-10))
    corr_denoised /= np.outer(d, d)
    np.fill_diagonal(corr_denoised, 1.0)
    return corr_denoised.astype(np.float32), n_signal

# Build sector adjacency for attention ratio evaluation
def build_adjacency(labels):
    L = labels.reshape(-1, 1)
    A = ((L == L.T) & (L != -1)).astype(np.float32)
    np.fill_diagonal(A, 0)
    return A
A_sector = build_adjacency(sectors)

valid_times = []
T_total = features.shape[0]
for t in range(LOOKBACK, T_total - HORIZON):
    c_now = features[t, :, 0]
    c_fut = features[t + HORIZON, :, 0]
    valid = ~((c_now == 0) | (c_fut == 0))
    if valid.sum() >= N_STOCKS * 0.5:
        valid_times.append(t)

n_total = len(valid_times)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)

print("Precomputing denoised correlation matrices...")
t0_cache = time.time()
unique_anchors = sorted(set(t // CORR_REUSE * CORR_REUSE for t in valid_times))
corr_cache = {}
for i, anchor in enumerate(unique_anchors):
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(unique_anchors)} anchors...")
    t_start = max(0, anchor - CORR_WINDOW)
    returns = features[t_start:anchor, :, 5]
    T_window = returns.shape[0]
    if T_window >= 20:
        corr = np.corrcoef(returns.T).astype(np.float32)
        corr = np.nan_to_num(corr, nan=0.0)
        corr, _ = mp_denoise(corr, T_window, N_STOCKS)
    else:
        corr = np.eye(N_STOCKS, dtype=np.float32)
    corr_cache[anchor] = corr
print(f"Cached {len(corr_cache)} matrices in {time.time() - t0_cache:.1f}s")

class StockDataset(Dataset):
    def __init__(self, features, times, corr_cache, corr_reuse=CORR_REUSE):
        self.features = features
        self.times = times
        self.corr_cache = corr_cache
        self.corr_reuse = corr_reuse
        self.N = features.shape[1]
    def __len__(self):
        return len(self.times)
    def __getitem__(self, idx):
        t = self.times[idx]
        x = self.features[t - LOOKBACK:t, :, 1:]
        x = np.nan_to_num(x, nan=0.0)
        c_now = self.features[t, :, 0]
        c_fut = self.features[t + HORIZON, :, 0]
        y_raw = np.where(c_now != 0, c_fut / c_now - 1, 0).astype(np.float32)
        ranks = rankdata(y_raw).astype(np.float32)
        y = (2.0 * (ranks - 1.0) / (self.N - 1.0) - 1.0).astype(np.float32)
        anchor = t // self.corr_reuse * self.corr_reuse
        corr = self.corr_cache[anchor]
        return (torch.from_numpy(x.transpose(1, 0, 2).copy()),
                torch.from_numpy(y.copy()),
                torch.from_numpy(corr))

train_ds = StockDataset(features, valid_times[:n_train], corr_cache)
val_ds   = StockDataset(features, valid_times[n_train:n_train + n_val], corr_cache)
test_ds  = StockDataset(features, valid_times[n_train + n_val:], corr_cache)
print(f"Samples: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## 3. Models: LSTM Baseline + DGT

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Baseline LSTM (identical to v3.0)
# ══════════════════════════════════════════════════════════════
class BaselineLSTM(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))
    def forward(self, x, corr=None):
        B, N, T, F = x.shape
        _, (h, _) = self.lstm(x.reshape(B * N, T, F))
        return self.head(h[-1]).squeeze(-1).reshape(B, N)


# ══════════════════════════════════════════════════════════════
#  Differential Graph Transformer (adapted from CS224W)
# ══════════════════════════════════════════════════════════════

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class DifferentialAttention(nn.Module):
    """Differential attention: softmax(Q1K1^T) - λ·softmax(Q2K2^T)
    Optionally conditioned on graph adjacency A."""
    def __init__(self, d_model, n_heads, depth=0):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads // 2
        self.scaling = self.head_dim ** -0.5
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        # Lambda parameters
        self.lambda_init = 0.8 - 0.6 * math.exp(-0.3 * depth)
        self.lambda_q1 = nn.Parameter(torch.zeros(self.head_dim).normal_(0, 0.1))
        self.lambda_k1 = nn.Parameter(torch.zeros(self.head_dim).normal_(0, 0.1))
        self.lambda_q2 = nn.Parameter(torch.zeros(self.head_dim).normal_(0, 0.1))
        self.lambda_k2 = nn.Parameter(torch.zeros(self.head_dim).normal_(0, 0.1))
        self.subln = RMSNorm(2 * self.head_dim)

    def forward(self, x, A=None, causal_mask=None):
        B, L, D = x.shape
        q = self.q_proj(x).view(B, L, 2 * self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, L, 2 * self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.n_heads, 2 * self.head_dim).transpose(1, 2)
        q = q * self.scaling
        attn = torch.matmul(q, k.transpose(-1, -2))
        attn = torch.nan_to_num(attn)
        if causal_mask is not None:
            attn = attn + causal_mask
        attn = F.softmax(attn, dim=-1, dtype=torch.float32).type_as(q)
        # Differential: split into positive and negative heads
        lambda_1 = torch.exp(torch.sum(self.lambda_q1 * self.lambda_k1)).type_as(q)
        lambda_2 = torch.exp(torch.sum(self.lambda_q2 * self.lambda_k2)).type_as(q)
        lambda_full = lambda_1 - lambda_2 + self.lambda_init
        attn = attn.view(B, self.n_heads, 2, L, L)
        diff_attn = attn[:, :, 0] * (1.0 if A is None else A) - lambda_full * attn[:, :, 1]
        out = torch.matmul(diff_attn, v)  # (B, n_heads, L, 2*head_dim)
        out = self.subln(out)
        out = out * (1 - self.lambda_init)
        out = out.transpose(1, 2).reshape(B, L, self.n_heads * 2 * self.head_dim)
        return out, diff_attn


class DGTLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.2, depth=0):
        super().__init__()
        self.temporal_attn = DifferentialAttention(d_model, n_heads, depth)
        self.spatial_attn = DifferentialAttention(d_model, n_heads, depth)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn1 = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(d_model * 2, d_model))
        self.ffn2 = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(d_model * 2, d_model))

    def forward(self, x, corr, causal_mask):
        # x: (B, N, T, D)
        B, N, T, D = x.shape
        # Temporal attention: per stock, across time (causal)
        xt = x.reshape(B * N, T, D)
        t_out, _ = self.temporal_attn(xt, causal_mask=causal_mask)
        xt = self.ln1(self.ffn1(t_out) + t_out)
        x = xt.reshape(B, N, T, D) + x
        # Spatial attention: per time step, across stocks (conditioned on corr)
        x = x.permute(0, 2, 1, 3).reshape(B * T, N, D)  # (B*T, N, D)
        A = corr.unsqueeze(1).expand(B, T, N, N).reshape(B * T, 1, N, N)  # broadcast across heads
        s_out, _ = self.spatial_attn(x, A=A)
        xs = self.ln2(self.ffn2(s_out) + s_out)
        x = xs.reshape(B, T, N, D).permute(0, 2, 1, 3) + x.reshape(B, T, N, D).permute(0, 2, 1, 3)
        return x  # (B, N, T, D)


class DGT(nn.Module):
    """Differential Graph Transformer (adapted from CS224W).
    Uses Pearson correlation for spatial attention conditioning."""
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=64, n_heads=4,
                 n_layers=2, T=LOOKBACK, N=N_STOCKS, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.time_emb = nn.Embedding(T, hidden_dim)
        self.stock_emb = nn.Embedding(N, hidden_dim)
        self.layers = nn.ModuleList([
            DGTLayer(hidden_dim, n_heads, dropout, depth=i)
            for i in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))
        self.T = T
        self.N = N
        # Pre-compute causal mask
        self.register_buffer('causal_mask',
            torch.triu(torch.full((T, T), float('-inf')), diagonal=1))

    def forward(self, x, corr=None):
        B, N, T, F = x.shape
        # Project features
        x = self.input_proj(x)  # (B, N, T, D)
        # Add time and stock embeddings
        t_emb = self.time_emb(torch.arange(T, device=x.device))  # (T, D)
        s_emb = self.stock_emb(torch.arange(N, device=x.device))  # (N, D)
        x = x + t_emb[None, None, :, :] + s_emb[None, :, None, :]
        # Expand causal mask for batched temporal attention
        cm = self.causal_mask.unsqueeze(0).expand(B * N, T, T).unsqueeze(1)  # (B*N, 1, T, T)
        # DGT layers
        for layer in self.layers:
            x = layer(x, corr, cm)
        # Use final time step for prediction
        out = x[:, :, -1, :]  # (B, N, D)
        return self.head(out).squeeze(-1)  # (B, N)


print(f"BaselineLSTM: {sum(p.numel() for p in BaselineLSTM().parameters()):,} params")
print(f"DGT:          {sum(p.numel() for p in DGT().parameters()):,} params")

## 4. Training

In [ ]:
def rank_ic(pred, actual):
    p = pred.detach().cpu().numpy().flatten()
    a = actual.detach().cpu().numpy().flatten()
    m = ~(np.isnan(p) | np.isnan(a))
    if m.sum() < 10: return 0.0
    c, _ = spearmanr(p[m], a[m])
    return c if not np.isnan(c) else 0.0

def ic_loss(pred, target):
    p = pred - pred.mean(dim=-1, keepdim=True)
    t = target - target.mean(dim=-1, keepdim=True)
    p_norm = p.norm(dim=-1) + 1e-8
    t_norm = t.norm(dim=-1) + 1e-8
    return -(p * t).sum(dim=-1) / (p_norm * t_norm)

def train_epoch(model, loader, opt, mt):
    model.train()
    tl, ti, nb = 0, 0, 0
    for x, y, corr in loader:
        x, y, corr = x.to(device), y.to(device), corr.to(device)
        opt.zero_grad()
        pred = model(x) if mt == "lstm" else model(x, corr)
        loss = ic_loss(pred, y).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item(); ti += rank_ic(pred, y); nb += 1
    return tl / nb, ti / nb

@torch.no_grad()
def evaluate(model, loader, mt):
    model.eval()
    tl, ti, nb = 0, 0, 0
    for x, y, corr in loader:
        x, y, corr = x.to(device), y.to(device), corr.to(device)
        pred = model(x) if mt == "lstm" else model(x, corr)
        tl += ic_loss(pred, y).mean().item(); ti += rank_ic(pred, y); nb += 1
    return tl / nb, ti / nb

def train_model(model, opt, ltr, lva, mt, epochs=150, patience=30, sched_patience=10):
    best_ic, best_st, wait = -np.inf, None, 0
    ic_history = []
    hist = {"train_loss": [], "val_loss": [], "train_ic": [], "val_ic": []}
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=sched_patience, min_lr=1e-6)
    blrs = [pg["lr"] for pg in opt.param_groups]
    for ep in range(epochs):
        if ep < 5:
            for pg, bl in zip(opt.param_groups, blrs):
                pg["lr"] = bl * (ep + 1) / 5
        tl, ti = train_epoch(model, ltr, opt, mt)
        vl, vi = evaluate(model, lva, mt)
        if ep >= 5: sched.step(vl)
        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        hist["train_ic"].append(ti); hist["val_ic"].append(vi)
        ic_history.append(vi)
        avg_ic = np.mean(ic_history[-5:]) if len(ic_history) >= 5 else np.mean(ic_history)
        if avg_ic > best_ic:
            best_ic = avg_ic
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if (ep + 1) % 10 == 0:
            print(f"  Ep {ep+1:3d} | TrL:{tl:.6f} TrIC:{ti:+.4f} | "
                  f"VaL:{vl:.6f} VaIC:{vi:+.4f} | best_avg:{best_ic:+.4f} "
                  f"lr:{opt.param_groups[0]['lr']:.1e}")
        if wait >= patience:
            print(f"  Early stop ep {ep+1} (best smoothed val IC: {best_ic:+.4f})")
            break
    if best_st: model.load_state_dict(best_st)
    print(f"  Done {len(hist['train_loss'])} ep, best smoothed val IC: {best_ic:+.4f}")
    return hist

print("Training utilities ready.")

In [ ]:
BATCH_SIZE = 16; LR = 1e-3; N_SEEDS = 20

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

all_results = []
all_histories = {"lstm": [], "dgt": []}
total_t0 = time.time()

for seed in range(N_SEEDS):
    seed_t0 = time.time()
    print(f"\n{'=' * 60}")
    print(f"  SEED {seed + 1}/{N_SEEDS}")
    print(f"{'=' * 60}")
    torch.manual_seed(seed * 42)
    np.random.seed(seed * 42)

    # ── Baseline LSTM ──
    print("\n  [LSTM]")
    m_lstm = BaselineLSTM().to(device)
    o_lstm = torch.optim.AdamW(m_lstm.parameters(), lr=LR, weight_decay=1e-4)
    h_lstm = train_model(m_lstm, o_lstm, train_loader, val_loader, mt="lstm")

    # ── DGT ──
    print("\n  [DGT]")
    m_dgt = DGT().to(device)
    o_dgt = torch.optim.AdamW(m_dgt.parameters(), lr=LR, weight_decay=1e-4)
    h_dgt = train_model(m_dgt, o_dgt, train_loader, val_loader, mt="dgt")

    # ── Test evaluation ──
    tl_l, ti_l = evaluate(m_lstm, test_loader, mt="lstm")
    tl_d, ti_d = evaluate(m_dgt, test_loader, mt="dgt")

    result = {"seed": seed, "lstm_loss": tl_l, "lstm_ic": ti_l,
              "dgt_loss": tl_d, "dgt_ic": ti_d}
    all_results.append(result)
    all_histories["lstm"].append(h_lstm)
    all_histories["dgt"].append(h_dgt)

    elapsed = time.time() - seed_t0
    print(f"\n  → LSTM IC: {ti_l:+.4f} | DGT IC: {ti_d:+.4f} | {elapsed:.0f}s")

total_time = time.time() - total_t0
print(f"\nAll seeds done in {total_time/60:.1f} min.")

## 5. Results

In [ ]:
lstm_ics = [r["lstm_ic"] for r in all_results]
dgt_ics = [r["dgt_ic"] for r in all_results]

print(f"{'=' * 65}")
print(f"  DGT BASELINE — {N_SEEDS} Seeds, N={N_STOCKS} stocks")
print(f"{'=' * 65}")
print(f"\n{'Seed':<6} {'LSTM IC':>10} {'DGT IC':>10}")
print(f"{'-' * 30}")
for r in all_results:
    print(f"{r['seed']:<6} {r['lstm_ic']:>+10.4f} {r['dgt_ic']:>+10.4f}")

print(f"\n{'=' * 65}")
print(f"  AGGREGATE")
print(f"{'=' * 65}")
print(f"LSTM IC: {np.mean(lstm_ics):+.4f} ± {np.std(lstm_ics):.4f}")
print(f"DGT  IC: {np.mean(dgt_ics):+.4f} ± {np.std(dgt_ics):.4f}")

dgt_wins = sum(1 for l, d in zip(lstm_ics, dgt_ics) if d > l)
print(f"\nDGT > LSTM in {dgt_wins}/{N_SEEDS} seeds")

if N_SEEDS >= 3:
    t_stat, p_val = ttest_rel(dgt_ics, lstm_ics)
    print(f"Paired t-test (DGT - LSTM): t={t_stat:.3f}, p={p_val:.3f}")

# Training curves (seed 0)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
bh_l = all_histories["lstm"][0]
bh_d = all_histories["dgt"][0]

axes[0].plot(bh_l["train_loss"], ls="--", alpha=0.6, label="LSTM Train")
axes[0].plot(bh_l["val_loss"], lw=2, label="LSTM Val")
axes[0].plot(bh_d["train_loss"], ls="--", alpha=0.6, label="DGT Train")
axes[0].plot(bh_d["val_loss"], lw=2, label="DGT Val")
axes[0].set(xlabel="Epoch", ylabel="IC Loss", title="Loss Curves (seed 0)")
axes[0].legend()

axes[1].plot(bh_l["val_ic"], lw=2, label="LSTM Val IC")
axes[1].plot(bh_d["val_ic"], lw=2, label="DGT Val IC")
axes[1].axhline(0, color="gray", ls=":", alpha=0.5)
axes[1].set(xlabel="Epoch", ylabel="Rank IC", title="Validation IC (seed 0)")
axes[1].legend()

x = np.arange(N_SEEDS); w = 0.35
axes[2].bar(x - w/2, lstm_ics, w, label="LSTM", color="#1f77b4", alpha=0.8)
axes[2].bar(x + w/2, dgt_ics, w, label="DGT", color="#2ca02c", alpha=0.8)
axes[2].axhline(0, color="gray", ls=":", alpha=0.5)
axes[2].set(xlabel="Seed", ylabel="Test Rank IC", title=f"IC Across {N_SEEDS} Seeds")
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/dgt_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
results_json = {
    "version": "dgt_baseline",
    "config": {
        "N_STOCKS": N_STOCKS, "INPUT_DIM": INPUT_DIM,
        "N_SEEDS": N_SEEDS, "LOOKBACK": LOOKBACK, "HORIZON": HORIZON,
        "BATCH_SIZE": BATCH_SIZE, "LR": LR,
        "target": "rank-normalized [-1, 1]",
        "loss": "ic_loss",
        "early_stopping": "smoothed val IC (5-epoch rolling average)",
        "graph": "Pearson correlation (MP denoised, 60-day rolling)",
        "architecture": "Differential Graph Transformer (CS224W)",
    },
    "aggregate": {
        "lstm_ic_mean": float(np.mean(lstm_ics)),
        "lstm_ic_std": float(np.std(lstm_ics)),
        "dgt_ic_mean": float(np.mean(dgt_ics)),
        "dgt_ic_std": float(np.std(dgt_ics)),
        "dgt_wins": dgt_wins,
    },
    "per_seed": all_results,
}

with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump(results_json, f, indent=2, default=str)
print(f"Saved to {OUT_DIR}/metrics.json")